
# SILVER LAYER - FHIR Condition Processing (Production Ready)



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
import logging
import json
from pyspark.sql.functions import (
    col, trim, lower, row_number
)
from pyspark.sql.window import Window

In [0]:
# ====================== Config Details ======================
CONFIG = {
    "resource": "Condition",
    "run_date": datetime.today().strftime("%Y-%m-%d"),
    "bronze_path": "fhir_data.prd_bronze",
    "silver_path": "fhir_data.prd_silver"
}

resource = CONFIG["resource"]
run_date = CONFIG["run_date"]
bronze_base = CONFIG["bronze_path"]
silver_base = CONFIG["silver_path"]

In [0]:
# ====================== LOGGING ======================
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

logger.info(f"Starting Silver Layer | Resource: {resource} | Run Date: {run_date}")

In [0]:
# ====================== SCHEMA ======================
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# ====================== MAIN FUNCTION ======================
def process_condition_silver(resource, run_date, bronze_base):

    bronze_path = f"{bronze_base}.{resource.lower()}"
    logger.info(f"Reading bronze data from: {bronze_path}")

    df = spark.read\
        .table(bronze_path) \
        .filter(col("load_date") == run_date)

    logger.info(f"Initial count: {df.count()}")
    df = df.dropDuplicates()
    #df.display()

    # ================= SCHEMA =================
    schema = StructType([
        StructField("id", StringType(), True),

        StructField("resourceType", StringType(), True),

        StructField("code", StructType([
            StructField("coding", ArrayType(
                StructType([
                    StructField("code",    StringType(), True),
                    StructField("display", StringType(), True),
                    StructField("system",  StringType(), True)
                ])
            ), True),
            StructField("text", StringType(), True)
        ]), True),

        StructField("subject", StructType([
            StructField("reference", StringType(), True)
        ]), True),

        StructField("encounter", StructType([
            StructField("reference", StringType(), True)
        ]), True),

        StructField("clinicalStatus", StructType([
            StructField("coding", ArrayType(
                StructType([
                    StructField("code",   StringType(), True),
                    StructField("system", StringType(), True)
                ])
            ), True)
        ]), True),

        StructField("onsetDateTime",    StringType(), True),
        StructField("recordedDate",     StringType(), True),

        StructField("meta", StructType([
            StructField("lastUpdated", StringType(), True),
            StructField("source",      StringType(), True),
            StructField("versionId",   StringType(), True)
        ]), True),
    ])

    df_parsed = df.withColumn("parsed", from_json(col("resource"), schema))

    # ======================== FLATTEN =========================
    df_silver = (
        df_parsed
        # Explode code.coding array — one row per coding entry
        .withColumn("coding", explode_outer(col("parsed.code.coding")))
        .select(
            col("parsed.id").alias("condition_id"),
            col("parsed.resourceType").alias("resource_type"),

            # code block
            col("coding.code").alias("condition_code"),
            col("coding.display").alias("condition_display"),
            col("coding.system").alias("coding_system"),
            col("parsed.code.text").alias("condition_text"),

            # subject → patient_id
            split(col("parsed.subject.reference"), "/")[1].alias("patient_id"),

            # encounter ref (optional — may be null)
            split(col("parsed.encounter.reference"), "/")[1].alias("encounter_id"),

            # clinical status (first element of array)
            col("parsed.clinicalStatus.coding")[0]["code"].alias("clinical_status"),

            # dates
            to_timestamp(col("parsed.onsetDateTime")).alias("onset_ts"),
            to_date(col("parsed.recordedDate")).alias("recorded_date"),

            # meta
            to_timestamp(col("parsed.meta.lastUpdated")).alias("last_updated_ts"),
            col("parsed.meta.source").alias("source"),
            col("parsed.meta.versionId").alias("version_id"),

            current_date().alias("ingestion_date")
        )
    )

    # ====================== CLEAN ======================
    df_clean = (
        df_silver
        .withColumn("condition_id",      trim(col("condition_id")))
        .withColumn("patient_id",         trim(col("patient_id")))
        .withColumn("encounter_id",       trim(col("encounter_id")))
        .withColumn("condition_code",     trim(col("condition_code")))
        .withColumn("condition_display",  trim(col("condition_display")))
        .withColumn("resource_type",      trim(col("resource_type")))
        .withColumn("clinical_status",    lower(trim(col("clinical_status"))))
    )
    #df_clean.display()

    # ====================== VALIDATE ======================
    df_valid = (
        df_clean
        .filter(col("condition_id").isNotNull())
        .filter(col("patient_id").isNotNull())
        .filter(col("condition_code").isNotNull())
        .filter(col("last_updated_ts").isNotNull())
        .filter(col("resource_type") == "Condition")
    )

    # Allow known SNOMED + ICD-10 systems only
    valid_systems = [
        "http://snomed.info/sct",
        "http://hl7.org/fhir/sid/icd-10",
        "http://hl7.org/fhir/sid/icd-10-cm"
    ]
    df_valid = df_valid.filter(col("coding_system").isin(valid_systems))

    # Allow known clinical statuses (or null — status is optional in Condition)
    valid_statuses = ["active", "recurrence", "relapse", "inactive", "remission", "resolved"]
    df_valid = df_valid.filter(
        col("clinical_status").isin(valid_statuses) | col("clinical_status").isNull()
    )

    # ====================== DEDUP ======================
    # Keep latest version per condition_id + condition_code combination
    window_spec = Window.partitionBy("condition_id", "condition_code") \
                        .orderBy(col("last_updated_ts").desc())

    df_dedup = (
        df_valid
        .withColumn("rn", row_number().over(window_spec))
        .filter(col("rn") == 1)
        .drop("rn")
    )
    df_dedup = df_dedup.withColumn(
    "dq_flags",
    concat_ws("|",
        when(col("clinical_status").isNull(), lit("MISSING_CLINICAL_STATUS")),
        when(col("encounter_id").isNull(),    lit("NO_ENCOUNTER_LINK")),
        when(col("onset_ts").isNull(),        lit("NO_ONSET_DATE")),
        when(col("recorded_date").isNull(),   lit("NO_RECORDED_DATE"))))\
            .withColumn("effective_start_date", current_timestamp()) \
    .withColumn("effective_end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

    #df_dedup.display()
    return df_dedup


#process_condition_silver(resource, run_date, bronze_base)

In [0]:
# ====================== Main EXECUTION ======================
try:
    df_silver = process_condition_silver(resource, run_date, bronze_base)
    df_silver.createOrReplaceTempView("cond_src")

    silver_path = f"{silver_base}.{resource.lower()}"
    spark.sql(f"""
              MERGE INTO {silver_path} t
USING cond_src s
ON t.condition_id = s.condition_id AND t.is_current = true

-- 1. Expire existing record if anything changed
WHEN MATCHED AND (
    NOT (t.resource_type      <=> s.resource_type) OR
    NOT (t.condition_code     <=> s.condition_code) OR
    NOT (t.condition_display  <=> s.condition_display) OR
    NOT (t.coding_system      <=> s.coding_system) OR
    NOT (t.condition_text     <=> s.condition_text) OR
    NOT (t.patient_id         <=> s.patient_id) OR
    NOT (t.encounter_id       <=> s.encounter_id) OR
    NOT (t.clinical_status    <=> s.clinical_status) OR
    NOT (t.onset_ts           <=> s.onset_ts) OR
    NOT (t.recorded_date      <=> s.recorded_date) OR
    NOT (t.last_updated_ts    <=> s.last_updated_ts) OR
    NOT (t.source             <=> s.source) OR
    NOT (t.version_id         <=> s.version_id) OR
    NOT (t.dq_flags           <=> s.dq_flags)
)
THEN UPDATE SET
    t.is_current = false,
    t.effective_end_date = current_timestamp()

--  2. Insert new or changed record
WHEN NOT MATCHED
THEN INSERT (
    condition_id,
    resource_type,
    condition_code,
    condition_display,
    coding_system,
    condition_text,
    patient_id,
    encounter_id,
    clinical_status,
    onset_ts,
    recorded_date,
    last_updated_ts,
    source,
    version_id,
    ingestion_date,
    dq_flags,
    effective_start_date,
    effective_end_date,
    is_current
)
VALUES (
    s.condition_id,
    s.resource_type,
    s.condition_code,
    s.condition_display,
    s.coding_system,
    s.condition_text,
    s.patient_id,
    s.encounter_id,
    s.clinical_status,
    s.onset_ts,
    s.recorded_date,
    s.last_updated_ts,
    s.source,
    s.version_id,
    s.ingestion_date,
    s.dq_flags,
    current_timestamp(),
    NULL,
    true
) 
              
              """)

    # df_silver.write.format("delta") \
    #     .mode("append") \
    #     .option("mergeSchema", "true") \
    #     .saveAsTable(silver_path)

    logger.info(f"Successfully written to Silver: {silver_path}")

    #df_silver.display()
    df_silver.printSchema()

except Exception as e:
    logger.error(f"Pipeline failed: {str(e)}", exc_info=True)
    raise

finally:
    logger.info("Pipeline execution completed")